# Derivative thresholds for DDS

* Read noise streams file and calculates derivative using DDS with several combinations of window and offset    
* Calculates RMS of noise derivative for each (w,o) combination


In [ ]:
import os
import pty
import matplotlib.pyplot as plt
from astropy.io import fits
import numpy as np
import tempfile
from subprocess import run
import auxpileup as aux
import argparse
import pandas as pd
import sys


In [ ]:
tmpDir = tempfile.mkdtemp()
os.environ["PFILES"] = f"{tmpDir}:{os.environ['PFILES']}"
os.environ["HEADASNOQUERY"] = ""
os.environ["HEADASPROMPT"] = "/dev/null/"

In [ ]:
# parameter handling
def get_parameters():
    """
    Get parameters for threshold computation in DDS window/offset analysis.
    If running in a Jupyter Notebook, use default parameters.
    If running as a script (e.g., SLURM), parse command line arguments.
    """
    

    # Check if running in a Jupyter Notebook or as a script 
    if aux.is_notebook():
        # Default parameters for interactive use
        print("Running in notebook mode for threshold analysis")
        return {
            "windows": [0,1, 2, 3, 4, 5, 6], # subtraction derivative window for detection
            "offsets": [0,1, 2, 3, 4, 5, 6],  # offset for subtraction window
            "init_rec": 1,  # initial noise record to process
            "final_rec": 2,  # final noise record to process
            "th_sigmas": [1.5, 2.,2.5,3.,3.5,4.,5., 6., 7.], # thresholds in sigmas for detection
            "SD": 2, # samples down in SIRENA
            "SU": 3, # samples up in SIRENA
            "false_detections_file": "tmp_fakes_rec1_rec2.fits",  # output CSV filename
            #"noise_rms_file": "tmp_noise_rms.csv"  # output CSV filename for noise RMS values
            "noise_rms_file": None  # output CSV filename for noise RMS values
        }
    else:
        # Parameters from command line (e.g., for SLURM)
        parser = argparse.ArgumentParser(
            description='Execute the python script for threshold analysis',
            prog='execute_sigma_wo_noise_derivative_analysis.py')
        parser.add_argument('--windows', type=int, nargs='*', default=[0, 1, 2, 3, 4, 5, 6, 10, 15, 20],
                            help='Subtraction derivative window for detection')
        parser.add_argument('--offsets', type=int, nargs='*', default=[0, 1, 2, 3, 4, 5, 6],
                            help='Offset for subtraction window')
        parser.add_argument('--init_rec', type=int, default=1,
                            help='Initial noise record to process')
        parser.add_argument('--final_rec', type=int, default=100,
                            help='Final noise record to process')
        parser.add_argument('--th_sigmas', type=float, nargs='*', 
                            default=[1,1.1,1.2,1.3,1.4,1.5,1.6,1.7,1.8,1.9, 2.,2.2,2.4,2.6,2.8, 3.,3.5,4.,4.5, 5., 6., 7.],
                            help='Thresholds in sigmas for detection')
        parser.add_argument('--SD', type=int, default=2,
                            help='Samples down in SIRENA')
        parser.add_argument('--SU', type=int, default=3,
                            help='Samples up in SIRENA')
        parser.add_argument('--false_detections_file', type=str, default=None,
                            help='Output CSV filename for false detections')
        parser.add_argument('--noise_rms_file', type=str, default=None,
                                    help='Output CSV filename for noise RMS values')
        args = parser.parse_args()
              
        return vars(args)

## Parameters

In [ ]:
params = get_parameters()
windows = params["windows"]
offsets = params["offsets"]
init_rec = params["init_rec"]
final_rec = params["final_rec"]
nnrecords = final_rec - init_rec + 1
th_sigmas = params["th_sigmas"]
SD = params["SD"]
SU = params["SU"]
false_detections_file = params["false_detections_file"]
noise_rms_file = params["noise_rms_file"]
if false_detections_file is None:
    false_detections_file = f"./analysis_pairs/threshold_analysis_results_rec{init_rec}_rec{final_rec}.fits"
if noise_rms_file is None:
    noise_rms_file = f"./analysis_pairs/noise_rms_values.csv"
if os.path.exists(noise_rms_file):
    make_rms_file = False
    print(f"RMS noise values will be read from file: {noise_rms_file}")
else:
    make_rms_file = True

In [ ]:
# print the parameters for verification
print(f"Parameters for threshold analysis:")
print(f"  Windows: {windows}")
print(f"  Offsets: {offsets}")
print(f"  Initial record: {init_rec}")
print(f"  Final record: {final_rec}")
print(f"  Number of records: {nnrecords}")
print(f"  Threshold sigmas: {th_sigmas}")
print(f"  Samples down (SD): {SD}")
print(f"  Samples up (SU): {SU}")
print(f"  False detections output file: {false_detections_file}")
print(f"  Noise RMS output file: {noise_rms_file}")

In [ ]:
noise_file = "/dataj6/ceballos/INSTRUMEN/EURECA/TN350_detection/2024_revision/analysis_pairs/noise_50000recs.fits"
kernel_coeficients = [1, 1, -1, -1]  # 1st derivative kernel
start_sample_for_kernel1 = 1 #[i=1, i=0, i=-1 , i=-2] # CEA/Saclay
lib_sirena = "/dataj6/ceballos/INSTRUMEN/EURECA/ERESOL/CEASaclay/May2025_v5_v20250621/optimal_filters_6keV_50x30.fits"
xml_xifusim = "config_xifu_50x30_v5_20250621.xml"
max_num_coefficients = 100  # maximum number of coefficients in the kernel, to accommodate the largest window and offset


## Utility functions

In [ ]:
from noise_analysis_utils import (
    mean_derivative,
    derive_new_kernel,
    derive_new_kernel_array,
    derivative_from_kernel,
    rms_from_kernel,
    save_fake_detections_fits,
    load_fake_detections_fits,
    get_detections,
)

## Read Noise file

In [ ]:
with fits.open(noise_file) as hdulist:
    tclock = 7.68E-6
    sampling_rate = 1.0 / tclock  # Hz
    records = hdulist["TESRECORDS"].data
    all_record_stream = records["ADC"]
    if final_rec > len(all_record_stream):
        final_rec = len(all_record_stream)


## Calculate derivative using DDS (win/=0) and baseline derivative (DDS with w=0)    

```d = 1* s_{i+1} + 1 * s_{i} + (-1)*s_{i-1} + (-1)*s_{i-2}```      # [1,1,-1,-1] DDS with w=0;o=0  (baseline)

In [ ]:
if make_rms_file:
    # create a numpy array to store a list of kernel coefficients for each combination of window and offset
    # third axis is the number of coefficients in the kernel, which is 100 to accommodate the largest window and offset
    kernel_coefficients_list = np.zeros((len(windows), len(offsets), 100), dtype=float)

### calculate kernel coefficients for each configuration

In [ ]:
if make_rms_file:
    for iw, window in enumerate(windows):
        for io, offset in enumerate(offsets):
            if window == 0 and offset > 0:
                continue  # skip this case since it is not defined
            kernel_coefficients_list[iw, io, :] = derive_new_kernel_array(
                kernel_coeficients, start_sample_for_kernel1, window, offset, max_num_coefficients
            )
    # do some testing to check that the kernel coefficients are correct
    if aux.is_notebook():
        print("Testing kernel coefficients for window=2, offset=1")
        print("kernel coefficients:", kernel_coefficients_list[2, 1, :])

### calculate RMS of each record

* calculate also mean RMS (using all noise records)   
* calculate also STD of RMS (using all noise records)   

In [ ]:
if make_rms_file:
    # rms_derivative[iw, io, record] = RMS of the (window, offset) derivative for that record stream
    rms_derivative = np.full((len(windows), len(offsets), all_record_stream.shape[0]), np.nan, dtype=float)
    mean_rms_derivative = np.full((len(windows), len(offsets)), np.nan, dtype=float)
    std_rms_derivative = np.full((len(windows), len(offsets)), np.nan, dtype=float)

    for iw, window in enumerate(windows):
        for io, offset in enumerate(offsets):
            if window == 0 and offset > 0:
                continue  # skip this case since it is not defined
            rms_derivative[iw, io, :] = rms_from_kernel(
                all_record_stream, kernel_coefficients_list[iw, io, :], start_sample_for_kernel1
            )
            mean_rms_derivative[iw, io] = np.nanmean(rms_derivative[iw, io, :])
            std_rms_derivative[iw, io] = np.nanstd(rms_derivative[iw, io, :])

### Save RMS results to a file

In [ ]:
if make_rms_file:
    # save the results to a CSV file for further analysis
    df = pd.DataFrame({
        'window': np.repeat(windows, len(offsets)),
        'offset': np.tile(offsets, len(windows)),
        'mean_rms': mean_rms_derivative.flatten(),
        'std_rms': std_rms_derivative.flatten(),
        'th6_sigmas': 6 / mean_rms_derivative.flatten()
    })
    df.to_csv(noise_rms_file, index=False)

In [ ]:
#sys.exit(0)  # exit the script after computing and saving the RMS values, since the main SIRENA loop is not needed for this part    

In [ ]:
if not make_rms_file:
    # noise_rms_file already existed, so the RMS computation above was skipped -- read
    # mean_rms_derivative/std_rms_derivative back from it instead, since the main SIRENA
    # loop below needs them regardless of whether they were just computed or cached.
    rms_df = pd.read_csv(noise_rms_file)
    expected_rows = len(windows) * len(offsets)
    if len(rms_df) != expected_rows:
        raise ValueError(f"File {noise_rms_file} exists but has {len(rms_df)} rows, expected {expected_rows}.")

    mean_rms_derivative = rms_df["mean_rms"].values.reshape(len(windows), len(offsets))
    std_rms_derivative = rms_df["std_rms"].values.reshape(len(windows), len(offsets))

In [ ]:
print("Finishing writing or reading RMS noise values from file:", noise_rms_file)

## Check Gaussian noise   
Check the noise is stationary and Gaussian enough. Histogram $d_{dds}$ against a Gaussian; check the autocorrelation out to lag ~ w+o+4. If there are non-Gaussian tails, the extrapolation in C2 (to derive iso-FAR thresholds) must be empirical rather than tail-model-based. Record the lag-2 autocorrelation — it explains why sigma ratios ($\sigma_{2,1}/\sigma_{0,0}$ ~ 2.3) exceed $\sqrt{1+1/w}=1.22$ and is worth a sentence in the paper (see OneNote notebook named SIRENA)

In [ ]:
# histogram of the derivative for a specific window and offset, to check if it is Gaussian
iw=3
window=windows[iw]
io=2
offset=offsets[io]
kernel_coefficients_list_iw_io = derive_new_kernel_array(
                kernel_coeficients, start_sample_for_kernel1, window, offset, max_num_coefficients
            )
single_record = all_record_stream[0:1]  # keep 2D shape (1, n_samples) for derivative_from_kernel
d_dds = derivative_from_kernel(
                single_record, kernel_coefficients_list_iw_io, start_sample_for_kernel1
            )[0]  # per-sample derivative values for this one record
hist, bin_edges = np.histogram(d_dds, bins=100, density=True)
plt.figure(figsize=(10, 6))
plt.bar(bin_edges[:-1], hist, width=np.diff(bin_edges), edgecolor='black', alpha=0.7)
plt.title(f"Histogram of derivative for window={windows[iw]}, offset={offsets[io]}")
plt.xlabel("Derivative value")
plt.ylabel("Probability density")
plt.show()

In [ ]:
if aux.is_notebook():
    for iw, window in enumerate(windows):
        for io, offset in enumerate(offsets):
            print(f"Window: {window}, Offset: {offset}, Mean RMS: {mean_rms_derivative[iw, io]:.6f}, Std RMS: {std_rms_derivative[iw, io]:.6f}")

# SIRENA reconstruction of NOISE streams   

* Use different detection thresholds: [1, 1.5, 2., 2.5, 3., 3.5, 4., 4.5, 5.]*sigma
* For each combination of window/offset, sigma is different   
* Reconstruct every noise stream record and calculate number of detections (they will be False detections)

Logic:   

```
for each noise_record:
    for each win:
        for each off:
            get mean_rms(win,off)
            for each th in [1, 1.5, 2., 2.5, 3., 3.5, 4., 4.5, 5.]*mean_rms
                run SIRENA (win, off, th) -> get detections=false_detections[win,off,nsigmas]


In [ ]:
# initialize array to store the (sigma) threshold for each combination of window, offset, and threshold
thresholds = np.zeros((len(windows), len(offsets), len(th_sigmas)), dtype=float)

# accumulators for per-fake arrival sample / energy, one entry per (record, window, offset, threshold) combo
fake_rows = []
fake_positions_list = []
fake_energies_list = []


In [ ]:
for record_num in range(init_rec, final_rec + 1):
    # run fselect to extract the record and save it to a temporary file
    tmp_record_filename = f"./analysis_pairs/tmp_record_{record_num}.fits"
    comm_fselect = (f"fselect infile={noise_file}+9 "
                    f" expr='#row=={record_num}'"
                    f" outfile={tmp_record_filename}"
                    f" clobber=yes")
    output_fselect = run(comm_fselect, shell=True, capture_output=True)
    assert output_fselect.returncode == 0, f"fselect failed to run:\n{comm_fselect}; \nReason: {output_fselect.stderr.decode()}"

    # read noise file to get init TIME of record
    with fits.open(tmp_record_filename) as hdulist:
        record_init_time = hdulist["TESRECORDS"].data["TIME"][0]

    for iw, window in enumerate(windows):
        for io, offset in enumerate(offsets):
            if window == 0 and offset > 0:
                continue  # skip this case since it is not defined
            for ith_sigma, th_sigma in enumerate(th_sigmas):
                thresholds[iw, io, ith_sigma] = th_sigma * mean_rms_derivative[iw, io]
                if aux.is_notebook():
                    print(f"Record: {record_num}, Window: {window}, Offset: {offset}, Threshold is {th_sigma} sigmas")
                tmp_false_detections_filename = f"./analysis_pairs/tmp_false_detections_{record_num}_{window}_{offset}_{th_sigma}.fits"
                comm = (f"tesrecons Recordfile='{tmp_record_filename}' "
                            f" TesEventFile={tmp_false_detections_filename}"
                            f" LibraryFile={lib_sirena}"
                            f" XMLFile={xml_xifusim}"
                            f" clobber=yes"
                            f" EnergyMethod=OPTFILT"
                            f" OFStrategy=BYGRADE"
                            f" filtEeV=6000"
                            f" OFNoise=NSD"
                            f" samplesDown={SD}"   #changed for new smoothed derivative (4 samples)
                            f" samplesUp={SU}"
                            f" threshold={thresholds[iw, io, ith_sigma]}"
                            f" windowSize={window}"
                            f" offset={offset}"
                        )
                #aux.vprint(f"Running {comm}")
                output_tesrecons = run(comm, shell=True, capture_output=True)
                assert output_tesrecons.returncode == 0, f"tesrecons failed to run:{comm}; \nReason: {output_tesrecons.stderr.decode()}"

                # read fake arrival sample (TIME rebased to record start) and energy (SIGNAL) of every detection.
                # memmap=False forces a real in-memory copy -- otherwise hdulist[1].data['SIGNAL'] is a view
                # backed by the mmap'd file, which would keep the file descriptor open for the rest of the run
                # even after os.remove() below, since fake_energies_list holds onto it until the script exits.
                with fits.open(tmp_false_detections_filename, memmap=False) as hdulist:
                    fakes_arrival_sample = (hdulist[1].data['TIME'] - record_init_time) / tclock
                    fakes_energy = hdulist[1].data['SIGNAL']

                fake_rows.append((record_num, window, offset, th_sigma))
                fake_positions_list.append(np.asarray(fakes_arrival_sample))
                fake_energies_list.append(np.asarray(fakes_energy))
                os.remove(tmp_false_detections_filename)
    os.remove(tmp_record_filename)

In [ ]:
print("Finished SIRENA reconstruction for all records, windows, offsets, and thresholds.")

## Save a file with detection results

In [ ]:
# save per-fake arrival sample and energy for every (record, window, offset, threshold) combination
save_fake_detections_fits(false_detections_file, fake_rows, fake_positions_list, fake_energies_list)

In [ ]:
if aux.is_notebook():
    total_fakes = sum(len(p) for p in fake_positions_list)
    print(f"Collected {len(fake_rows)} (record, window, offset, threshold) combinations, {total_fakes} fake detections total")


In [ ]:
print("Finished notebook execution")